In [33]:
# This for 1st extracting only BOM image from either .pdf or .jpg drawings and then extracting the BOM material
# information in an excell file. The input is the name of the drawing file containing drawings.
# This .ipynb file was created on 20/3/2025 and final edited on 21/3/2025 to include .pdf drawing files.

In [64]:
import numpy as np
import math
import pandas as pd
import cv2
import os
import re
import tqdm
from scipy.io import loadmat
from pdf2image import convert_from_path

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder

from PIL import Image
import pytesseract

import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from keras import backend as K

#from utils import *

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from keras.callbacks import ModelCheckpoint
from keras.optimizers import Adam, SGD
from keras.layers import *

from keras.applications import MobileNetV2
from keras.applications import InceptionResNetV2

from keras.models import Model
from keras.models import model_from_json

from keras.models import load_model

from ultralytics import YOLO
import difflib # for fuzzy logic for closest match between strings


In [51]:
## Extracting BOM image from the test drawings with .jpg files using trained model :

def extract_bom_jpg(drg_folder):

    # Load the trained YOLOv8 model
    #model = YOLO("runs/detect/train9/weights/best.pt")
    model = YOLO("/Users/subrata/workstation/jupyterFiles/yolo_data_file/yolov8_dir/my_yolo_model_1.pt")

    # Input Path to drawing folder (containing drawing images in .jpg)
    drawing_folder = drg_folder

    # Output Folder inside same drawing folder to save cropped images
    output_folder = drg_folder + "cropped_bom_jpg/"
    os.makedirs(output_folder, exist_ok=True)  # Create folder if not exists

    # List of drawing images where 'bom' could not be detected
    no_bom_drgs = []

    # Process each image in the test folder
    for filename in os.listdir(drawing_folder):
        if filename.lower().endswith((".jpg", ".jpeg", ".png")):  # Check for image files
            drawing_image = os.path.join(drawing_folder, filename)

            # Run inference
            results = model(drawing_image, conf=0.5)  ## Run YOLO INFERENCE
            x1 = 0
            y1 = 0
            x2 = 0
            y2 = 0
            crop_count = 0

            # Load the image using OpenCV
            image = cv2.imread(drawing_image)
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # Convert BGR to RGB

            # Extract filename (without extension)
            image_name = os.path.splitext(filename)[0]

            # Process detection results
            for result in results:
                for box in result.boxes:
                    x1, y1, x2, y2 = map(int, box.xyxy[0])  # Bounding box coordinates
                    class_id = int(box.cls[0])  # Class index
                    confidence = float(box.conf[0])  # Confidence score

                    # Crop the detected object
                    cropped_obj = image[y1:(y2+10), x1:(x2+10)] # To increase cropped image size at right and bottom by 10 pixels

                    # Save the cropped image
                    cropped_filename_original = f"{output_folder}/{image_name}_crop_{crop_count}.jpg"
                
                    cropped_filename = re.sub(r'_crop_\d+', '', cropped_filename_original)

                    cv2.imwrite(cropped_filename, cv2.cvtColor(cropped_obj, cv2.COLOR_RGB2BGR))  # Convert RGB to BGR for saving
                    crop_count += 1

            # If no 'bom' is detected, add the image to the list
            if x1 == 0:
                no_bom_drgs.append(filename)
                print(f"No 'bom' detected in {filename}.")

    # Save the list of 'no bom' images to a text file inside the same drawing folder :
    no_bom_file = drg_folder + "no_bom_jpg.txt"
    with open(no_bom_file, "w") as f:
        for img in no_bom_drgs:
            f.write(img + "\n")

    print(f"\n📂 List of 'no bom' images saved to {no_bom_file}")
    print(f"📂 Cropped images saved in {output_folder}")

    return output_folder


In [52]:
## Extracting BOM from the test drawings with .pdf files using trained model :

def extract_bom_pdf(drg_folder):

    ## Load the trained YOLOv8 model
    model = YOLO("runs/detect/train9/weights/best.pt")

    # Input Path to drawing folder (containing drawing images in .pdf)
    drawing_folder = drg_folder
    output_folder = drg_folder + "cropped_bom_jpg/"  # Folder to save cropped images
    os.makedirs(output_folder, exist_ok=True)  # Create folder if not exists

    # List of PDF drawings where 'bom' could not be detected
    no_bom_drgs = []

    # Process each PDF in the test folder
    for filename in os.listdir(drawing_folder):
        if filename.lower().endswith(".pdf"):  # Check for PDF files
            pdf_path = os.path.join(drawing_folder, filename)

            # Convert PDF to images
            images = convert_from_path(pdf_path, dpi=300)  # Higher DPI for better OCR and detection

            pdf_has_bom = False  # Flag to track if BOM is detected in any page

            # Process each page
            for page_num, image in enumerate(images):
                # Convert PIL image to OpenCV format
                image_cv = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)

                # Run YOLO inference
                results = model(image_cv, conf=0.5)

                x1, y1, x2, y2 = 0, 0, 0, 0
                crop_count = 0

                # Process detection results
                for result in results:
                    for box in result.boxes:
                        x1, y1, x2, y2 = map(int, box.xyxy[0])  # Bounding box coordinates
                        class_id = int(box.cls[0])  # Class index
                        confidence = float(box.conf[0])  # Confidence score

                        # Crop the detected object
                        cropped_obj = image_cv[y1:(y2+10), x1:(x2+10)] # To increase cropped image size at right and bottom by 10 pixels

                        # Save the cropped image
                        cropped_filename = f"{output_folder}/{os.path.splitext(filename)[0]}_crop_{crop_count}.jpg"
                        cv2.imwrite(cropped_filename, cropped_obj)  # Save as BGR format
                        crop_count += 1
                        pdf_has_bom = True  # Mark that BOM was found

            # If no BOM was detected in any page, add to the list
            if not pdf_has_bom:
                no_bom_drgs.append(filename)
                #print(f"No 'bom' detected in {filename}.")

    # Save the list of PDFs with no BOM detected
    no_bom_file = drg_folder + "no_bom_jpg.txt"
    with open(no_bom_file, "w") as f:
        for pdf in no_bom_drgs:
            f.write(pdf + "\n")

    print(f"\n📂 List of 'no bom' PDFs saved to {no_bom_file}")
    print(f"📂 Cropped images saved in {output_folder}")

    return output_folder


In [53]:
## Extract initial all box sizes of the table structure of cropped BOM images :

def all_boxes(pred_bom):
    
    boxes_list = []
    boxes_other_list = []
    pred_bom_gray = cv2.cvtColor(pred_bom, cv2.COLOR_BGR2GRAY)

    ##thresholding the image to a binary image
    thresh,img_bin = cv2.threshold(pred_bom_gray,0,255,cv2.THRESH_BINARY | cv2.THRESH_OTSU)

    #inverting the image 
    img_bin_invert = 255-img_bin
      
    # Length(width) of kernel as 100th of total width
    # kernel_len = pred_bom_gray.shape[1]//100
    # added on 19/3/2025
    kernel_len = 25

    ## Defining a vertical kernel to detect all vertical lines of image 
    ver_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, kernel_len))

    # Defining a horizontal kernel to detect all horizontal lines of image
    hor_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (kernel_len, 1))

    #Use vertical kernel to detect and save the vertical lines in a jpg
    image_1 = cv2.erode(img_bin_invert, ver_kernel, iterations=3)
    vertical_lines = cv2.dilate(image_1, ver_kernel, iterations=3)

    #Use horizontal kernel to detect and save the horizontal lines in a jpg
    image_2 = cv2.erode(img_bin_invert, hor_kernel, iterations=3)
    horizontal_lines = cv2.dilate(image_2, hor_kernel, iterations=3)
    
    # Combine horizontal and vertical lines in a new third image, with both having same weight.
    img_vh = cv2.addWeighted(vertical_lines, 0.5, horizontal_lines, 0.5, 0.0)
    
    # A kernel of 2x2
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
    img_vh_d = cv2.dilate(img_vh, kernel, iterations=2)

    # Defining the cell boxes
    contours, hierarchy = cv2.findContours(img_vh_d, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

    boxes = np.zeros((len(contours), 4))

    for i in range(len(contours)):
    
        cnt = contours[i]
        x, y, w, h = cv2.boundingRect(cnt)
        boxes[i, 0] = x
        boxes[i, 1] = y
        boxes[i, 2] = w
        boxes[i, 3] = h

    boxes_list.append(boxes)
    
    return boxes_list, img_vh, img_vh_d   ## in format x, y, w, h. and boxes list is the list of horizontal lines in between 
                                            ## all vertical lines.


In [54]:
## For a valid box, same y value should appear at least 3 times for 'Sl.No.', 'Qty' and 'Description' columns.
## Also no y value of '0' is considered for valid boxes.

def filter_arrays(array_list):
    filtered_list = []  # List to store filtered arrays

    for arr in array_list:
        # Ensure the array is not empty
        if arr.size == 0:
            continue

        # Step 1: Identify y-values appearing at least 3 times
        y_values = arr[:, 1]
        unique_y, counts = np.unique(y_values, return_counts=True)

        # Keep only y-values appearing at least 3 times
        valid_y_values = {y for y, c in zip(unique_y, counts) if c >= 3}

        # Filter rows with these valid y-values
        filtered_arr = np.array([row for row in arr if row[1] in valid_y_values])

        # Step 2: Find the last occurrence of y = 0
        if filtered_arr.size > 0:
            last_zero_index = np.where(filtered_arr[:, 1] == 0)[0]
            if last_zero_index.size > 0:
                last_zero_index = last_zero_index[-1]  # Last occurrence index
                filtered_arr = filtered_arr[last_zero_index + 1:]  # Keep rows after

        # Add to the final list only if it's non-empty
        if filtered_arr.size > 0:
            filtered_list.append(filtered_arr)  # Corrected the append operation

    return filtered_list  # Return the list of filtered arrays


In [55]:
def find_bottom_left_one_rev(bom_boxes, pred_bom):

    """
    Find the bottom-most left-most box containing '1'
    """

    boxes_1 = filter_arrays(bom_boxes)
    boxes = sorted(boxes_1[0], key=lambda b: (b[1], b[0]))  # Sort by y (bottom-first), then x (left-first)

    # Convert list of arrays into a NumPy array
    boxes_np = np.array(boxes)

    # Step 1: Find the minimum x-value
    min_x = np.min(boxes_np[:, 0])  # Get minimum x

    # Step 2: Get all boxes with this min_x value
    filtered_boxes = boxes_np[boxes_np[:, 0] == min_x]
    
    for box in filtered_boxes:
        
        x, y, w, h = map(int, box)  # Convert to int
        
        x_c = x + 2  # These are to increase accuracy of reading text inside cell boxes
        y_c = y + 2  # These are to increase accuracy of reading text inside cell boxes
        w_c = w - 4  # These are to increase accuracy of reading text inside cell boxes
        h_c = h - 4  # These are to increase accuracy of reading text inside cell boxes

        pred_bom_gray = cv2.cvtColor(pred_bom, cv2.COLOR_BGR2GRAY)
        roi = pred_bom_gray[y_c:y_c+h_c, x_c:x_c+w_c]  # Crop the region - increase of x,y and reduction of w,h is very 

        #important for text accuracy
        sharp_kernel = np.array([[0, -1, 0], [-1, 5,-1], [0, -1, 0]])
        roi_sharp = cv2.filter2D(roi, -1, sharp_kernel)
        _, cell_bin = cv2.threshold(roi_sharp, 150, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)
        
        # OCR for text extraction :
        text = pytesseract.image_to_string(cell_bin, config='--psm 6 --oem 3').strip()  # OCR
        
        if text == '1':
            return (x, y, w, h)  # Return coordinates of this box
    
    return None  # If not found

def draw_horizontal_line_if_needed(pred_bom, x, y, w, h):
    """
    Check if the longest horizontal line below the given box has another such line below it.
    If not, draw a new horizontal line at the bottom of the image.

    :param image_path: Path to the input image
    :param x, y, w, h: Coordinates of the bottom-most-left-most box containing "1"
    :return: Modified image
    """
    
    # Convert image to grayscale
    image = cv2.cvtColor(pred_bom, cv2.COLOR_BGR2GRAY)
    
    # Apply binary threshold
    _, img_bin = cv2.threshold(image, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)
    
    # Invert image (white background, black lines)
    img_bin_inv = 255 - img_bin
    
    # Define horizontal kernel
    kernel_len = image.shape[1] // 100
    hor_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (kernel_len, 1))

    # Extract horizontal lines
    hor_lines = cv2.erode(img_bin_inv, hor_kernel, iterations=3)
    hor_lines = cv2.dilate(hor_lines, hor_kernel, iterations=3)

    # Find contours of horizontal lines
    contours, _ = cv2.findContours(hor_lines, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Convert contours to (x, y, w, h) format
    hor_lines_list = [cv2.boundingRect(cnt) for cnt in contours]

    # Sort lines based on the y-coordinate (bottom-most first)
    hor_lines_list = sorted(hor_lines_list, key=lambda x: x[1], reverse=True)

    # Find the longest horizontal line immediately below this box
    # Find the smallest y-value that is greater than y
    filtered_values = [item for item in hor_lines_list if item[1] > y]

    # Get the entry with the minimum y-value
    if filtered_values:
        longest_line = min(filtered_values, key=lambda x: x[1]) 
        
    if longest_line:
        x_longest, y_longest, w_longest, _ = longest_line

        # Check if another horizontal line exists below it with equal or greater width
        for x_line, y_line, w_line, h_line in hor_lines_list:
            if y_line > y_longest and w_line >= w_longest:
                #print("A longer or equal horizontal line exists below. No need to draw a new line.")
                return pred_bom  # No need to draw a new line

        # If no such line exists, draw a new horizontal line
        #print("No longer horizontal line found. Drawing a new line at the bottom.")
        # img_color = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
        img_height, img_width = pred_bom.shape[:2]
        y_new_line = img_height - 2 # two pixel above the bottom edge
        cv2.line(pred_bom, (0, y_new_line), (img_width, y_new_line), (0, 0, 0), 2)
        
        return pred_bom

    else:
        # print("No horizontal line found below the given box.")
        return pred_bom  # No modifications needed


In [59]:
## Take out header boxes and all other box info for all items separately :

def header_and_item(pred_bom_corrected):
    
    bom_boxes_first,_,_ = all_boxes(pred_bom_corrected)
    bom_boxes = filter_arrays(bom_boxes_first)

    for i in range(len(bom_boxes)):   # for single bom drawings, len(bom_boxes) is always 1

        if i == 0:
            col_count_list = []
            header_boxes_list = []
            bom_boxes_list = []
            item_index_width_list = []

        boxes_1 = bom_boxes[0]
        
        ## Find number of columns i.e. max occurance of any y value :

        y_list = boxes_1[:, 1].tolist()   ## since boxes are in x,y,w,h format (output from cv2.boundingRect)

        col_count_array = np.bincount(y_list)  ## no. of occurances of each y-value
        col_count = col_count_array.max()  ## max. no. of occurances of any one y-value

        # Now find the Header row :

        # In drawings, header row is always at the bottom.
        # Output of cv2.boundingRect always starts with the box of complete BOM
        # After this 1st row, the output starts with the bottom_most_right_most (in this order) box details.

        # So, y value of boxes[1] is the bottom_most-right_most (in this order) cell y value.

        y_bmrm_cell = boxes_1[0][1]  # y value of 1st bottom most cell

        # If occurances of this y value of 1st bottom most cell = number of columns, then this is the y value of Header Row

        num_col = y_list.count(y_bmrm_cell)   ## no. of occurances of y_bmrm_cell value 

        if num_col == col_count:
    
            y_header_row = y_bmrm_cell
            item_index_width = boxes_1[col_count-1, 2]  # This is the width of the 1st column i.e. 'Item No.' column
            bom_boxes_1 = boxes_1[1:]   ## remove 1st header row for bom consideration
    
        else:
        
            y_next_cell = boxes_1[1][1]

            num_col = y_list.count(y_next_cell)

            if num_col == col_count:
    
                y_header_row = y_next_cell
                item_index_width = boxes_1[col_count, 2]
                bom_boxes_1 = boxes_1[2:]    ## remove 1st 2 rows for bom consideration

        header_boxes = boxes_1[boxes_1[:, 1] == y_header_row] #######*************#########
    
        bom_boxes = bom_boxes_1[bom_boxes_1[:, 1] != y_header_row]
        
        col_count_list.append(col_count)
        item_index_width_list.append(item_index_width)
        header_boxes_list.append(header_boxes)
        bom_boxes_list.append(bom_boxes)
    
    return col_count_list, item_index_width_list, header_boxes_list, bom_boxes_list  ## all boxes in x, y, w, h format


In [42]:
# Function to standardize column names using difflib
def standardize_headers(df, reference_headers):
    standardized_cols = {
        col: difflib.get_close_matches(col, reference_headers, n=1, cutoff=0.6)[0] if difflib.get_close_matches(col, reference_headers, n=1, cutoff=0.6) else col
        for col in df.columns
    }
    return df.rename(columns=standardized_cols)


In [62]:
def bom_df_rev(pred_bom_id):

    pred_bom = cv2.imread(pred_bom_id)

    if pred_bom is None:
        print("Error: Image not found or could not be loaded.")
        return None, None, None

    boxes_list, _, _ = all_boxes(pred_bom)

    result = find_bottom_left_one_rev(boxes_list, pred_bom)
    if result:  # Check if function returned a valid box
        x, y, w, h = result
        #print(x, ',', y, ',', w, ',', h)
    else:
        print("No suitable box found")

    # Run function and get modified image
    pred_bom_corrected = draw_horizontal_line_if_needed(pred_bom, x, y, w, h)
    
    # Extract header and item details
    col_count_list, item_index_width_list, header_boxes_list, bom_boxes_list = header_and_item(pred_bom_corrected)
    
    # Ensure we have header boxes
    if not header_boxes_list or len(header_boxes_list) == 0:
        #print("Error: No header boxes found!")
        return None, None, None

    # Extract header column widths
    width_list = header_boxes_list[0][:, 2]  
    x_list = header_boxes_list[0][:, 0]
    
    # Find the frequency of column widths
    width_frequency = np.unique(width_list, return_counts=True)[1]  

    # Get all column widths from BOM items
    all_width_list = bom_boxes_list[0][:, 2].tolist()
    row_count_array = np.bincount(all_width_list)  
    row_count_apparent = row_count_array.max()
    
    # Estimate row count
    row_count = row_count_apparent // np.max(width_frequency) if np.max(width_frequency) != 0 else 0

    sharp_kernel = np.array([[0, -1, 0], [-1, 5,-1], [0, -1, 0]])

    # Extract header text
    header_text = []
    for box in header_boxes_list[0]:  
        x, y, w, h = map(int, box)  # Convert to integers
        
        x_c = x + 2
        y_c = y + 2
        w_c = w - 4
        h_c = h - 4

        pred_bom_gray = cv2.cvtColor(pred_bom_corrected, cv2.COLOR_BGR2GRAY)
        roi = pred_bom_gray[y_c:y_c+h_c, x_c:x_c+w_c]  # Crop the region - increase of x,y and reduction of w,h is very 
                                                       # important for text accuracy
        roi_sharp = cv2.filter2D(roi, -1, sharp_kernel)

        text = pytesseract.image_to_string(roi_sharp, config='--psm 6 --oem 3').strip()  # OCR
        text = text.replace("\n", " ")  # Remove newlines ***** added on 20/3/2025
        header_text.append(text)

    header_text.append('drg_id')
    
    serialised_header_text = header_text[::-1]  # Reverse order
    
    # Extract BOM items
    bom_text = []
    for i, bom_boxes in enumerate(bom_boxes_list):
        y_unique = np.unique(bom_boxes[:, 1])[::-1]  # Reverse order
        
        for j in y_unique:
            bom_sub_text = []
            aa = np.ones((col_count_list[0], 4))  
            bb = bom_boxes[bom_boxes[:, 1] == j]  # Select only rows with matching Y
            
            for k in range(len(aa)):
                for l in range(len(bb)):
                    if abs((bb[l][2] + bb[l][0]) - (width_list[k] + x_list[k])) <= 2:
                        aa[k, :] = bb[l, :]
                
            for k in range(len(aa)):
                x, y, w, h = map(int, aa[k])

                if x == 1 and y == 1 and w == 1 and h == 1:
                   text = ' '  
                else:
                    x_c = x + 2
                    y_c = y + 2
                    w_c = w - 4
                    h_c = h - 4

                    cell_crop_image = pred_bom_gray[y_c:y_c+h_c, x_c:x_c+w_c]

                    if cell_crop_image is None or cell_crop_image.size == 0:
                        print("Error: cell_crop_image is empty or not loaded correctly.")

                    # Find the first column where any value < 10
                    col_indices = np.where(np.any(cell_crop_image < 10, axis=0))[0]  # Get all matching column indices

                    if col_indices.size > 0:  # Check if any column meets the condition
                        col_index = col_indices[0]  # First column where condition is met
                        start_col = max(0, col_index - 3)  # Keep 3 columns before
                        modified_arr = cell_crop_image[:, start_col:]
                    else:
                        modified_arr = cell_crop_image  # Keep the original array unchanged

                    # OCR text extraction
                    cell_crop_sharp = cv2.filter2D(modified_arr, -1, sharp_kernel)
                    _, cell_bin = cv2.threshold(cell_crop_sharp, 150, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)
        
                    #text = pytesseract.image_to_string(roi, config='--psm 6').strip()  # OCR
                    text = pytesseract.image_to_string(cell_bin, config='--psm 6 --oem 3').strip()
            
                    text = text.replace("\n", " ")  # Remove newlines within extracted text
                    
                # OCR text extraction
                bom_sub_text.append(text)

            bom_id = ((pred_bom_id.split('/'))[-1].split('.'))[0] # making bom df with only drg no. and not with drg. path
            bom_sub_text.append(bom_id)  # Append image ID
            bom_text.append(bom_sub_text[::-1])  # Reverse order

    # Convert to DataFrame
    expected_cols = len(header_text)  # Expected column count
    filtered_bom_text = [row for row in bom_text if len(row) == expected_cols]

    # Convert to DataFrame
    df = pd.DataFrame(filtered_bom_text, columns=serialised_header_text)

    #df = pd.DataFrame(bom_text, columns=serialised_header_text)
    return df, bom_text, bom_sub_text


In [66]:
# Cropping and BOM printing final module :

folder_name = input("Enter folder name: ")
drg_folder = f"/Users/subrata/Desktop/{folder_name}/"

#+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
for file_name in os.listdir(drg_folder):
    if file_name.endswith((".jpg", ".png", ".jpeg")):
        output_cropped_folder = extract_bom_jpg(drg_folder)
    else:
        output_cropped_folder = extract_bom_pdf(drg_folder)
#++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

#Extract only bom part of the drawings :
#output_cropped_folder = extract_bom_jpg(drg_folder)

# Path to the folder containing images
folder_path = output_cropped_folder # Change this to your folder path

# List to store DataFrames
df_list = []

# Loop through all images in the folder
for file_name in os.listdir(folder_path):
    if file_name.endswith((".jpg", ".png", ".jpeg")):  # Add other image formats if needed
        image_path = os.path.join(folder_path, file_name)

        df, _, _ = bom_df_rev(image_path)  # Call your function

        if df is not None:  # Ensure df is valid
            bom_id = ((image_path.split('/'))[-1].split('.'))[0] # making bom df with only drg no. and not with drg. path
            print('BOM Created for = ', bom_id)

            df_list.append(df)

if df_list:

    # Take the header of the first DataFrame as the reference
    reference_headers = df_list[0].columns.tolist()

    # Apply standardization to all DataFrames (excluding the first, as it's already correct)
    df_list = [df_list[0]] + [standardize_headers(df, reference_headers) for df in df_list[1:]]

    # Concatenate all DataFrames
    final_df = pd.concat(df_list, ignore_index=True)
    final_df = final_df.dropna(axis=1, how="all")  # Drop completely empty columns
    print('ALL BOM CREATED SUCCESSFULLY')

else:
    print("No valid DataFrames generated.")

#final_df.to_excel("/Users/subrata/workstation/jupyterFiles/yolo_data_file/pred_cropped_bom_jpg/project_1.xlsx", index=False)  # Saves without the index column

# Define column indexes to sort by (e.g., 1st and 3rd columns)
sort_columns_1 = [final_df.columns[0]]  # Sort by 0th column
sort_columns_2 = [final_df.columns[3], final_df.columns[2]]  # Sort by 3rd and 2nd column

# Sort DataFrame
final_df_sorted_1 = final_df.sort_values(by=sort_columns_1, ascending=True)
final_df_sorted_2 = final_df.sort_values(by=sort_columns_2, ascending=True)

# Save both original and sorted DataFrame in the same Excel file
output_bom_file = drg_folder + "bill_of_material.xlsx"
#with pd.ExcelWriter("/Users/subrata/workstation/jupyterFiles/yolo_data_file/pred_cropped_bom_jpg/project_1.xlsx") as writer:
with pd.ExcelWriter(output_bom_file) as writer:
    final_df_sorted_1.to_excel(writer, sheet_name="Original Data", index=False)
    final_df_sorted_2.to_excel(writer, sheet_name="Sorted Data", index=False)

print("Bill of Material file saved successfully with two sheets!")





image 1/1 /Users/subrata/Desktop/project_1_jpg/SM-MFD-5KW-005-23.jpg: 480x640 1 bom, 58.8ms
Speed: 3.2ms preprocess, 58.8ms inference, 0.6ms postprocess per image at shape (1, 3, 480, 640)

image 1/1 /Users/subrata/Desktop/project_1_jpg/SM-MFD-5KW-001-23.jpg: 480x640 1 bom, 56.9ms
Speed: 2.1ms preprocess, 56.9ms inference, 0.5ms postprocess per image at shape (1, 3, 480, 640)

image 1/1 /Users/subrata/Desktop/project_1_jpg/SM-MFD-5KW-003-23.jpg: 480x640 1 bom, 54.8ms
Speed: 2.4ms preprocess, 54.8ms inference, 0.4ms postprocess per image at shape (1, 3, 480, 640)

image 1/1 /Users/subrata/Desktop/project_1_jpg/SM-MFD-5KW-006-23.jpg: 480x640 1 bom, 50.3ms
Speed: 1.9ms preprocess, 50.3ms inference, 0.3ms postprocess per image at shape (1, 3, 480, 640)

image 1/1 /Users/subrata/Desktop/project_1_jpg/SM-MFD-5KW-004-23.jpg: 480x640 (no detections), 49.4ms
Speed: 2.0ms preprocess, 49.4ms inference, 0.2ms postprocess per image at shape (1, 3, 480, 640)
No 'bom' detected in SM-MFD-5KW-004-23.j